# CUB concept-bottleneck pipeline: Le Conte Sparrow vs. Savannah Sparrow

Loads everything from the [`NWeak/cub-mirror`](https://huggingface.co/datasets/NWeak/cub-mirror) HuggingFace dataset (images, CLIP embeddings, official CUB labels/concepts) instead of local CQA caches, so this notebook runs the same way for anyone with `uv sync` + `hf auth login` -- no local CUB download needed.

Pipeline:
1. Train a CLIP-embedding -> concept classifier (`concept_mask`, 6 concepts) on all classes *except* the hard pair.
2. Train a concept -> label classifier on ground-truth concepts, but only on the hard pair (**Le Conte Sparrow** vs **Savannah Sparrow**, ids 123/126 -- two visually similar species).
3. Chain the two for an end-to-end (image -> predicted concepts -> predicted label) evaluation.
4. Save a tidy per-sample CSV of ground truth vs. predicted concepts and label.
5. Sanity-check a hand-computable linear formula (for the user study) against the fitted model's predictions.

In [1]:
from datasets import load_dataset

REPO_ID = "NWeak/cub-mirror"
ds = load_dataset(REPO_ID)
ds

/home/nicola.debole/projects/user-study-CBMs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['sample_idx', 'split', 'label', 'class_name', 'image_path', 'dagger beak', 'hooked seabird beak', 'all-purpose beak', 'cone beak', 'brown wing', 'grey wing', 'yellow wing', 'black wing', 'white wing', 'buff wing', 'brown upperparts', 'grey upperparts', 'yellow upperparts', 'black upperparts', 'white upperparts', 'buff upperparts', 'brown underparts', 'grey underparts', 'yellow underparts', 'black underparts', 'white underparts', 'buff underparts', 'solid breast', 'striped breast', 'multi-colored breast', 'brown back', 'grey back', 'yellow back', 'black back', 'white back', 'buff back', 'notched tail', 'brown upper-tail', 'grey upper-tail', 'black upper-tail', 'white upper-tail', 'buff upper-tail', 'eyebrow head', 'plain head', 'brown breast', 'grey breast', 'yellow breast', 'black breast', 'white breast', 'buff breast', 'grey throat', 'yellow throat', 'black throat', 'white throat', 'buff throat', 'black eye', 'beak length about the

## Pull out class/concept names and per-split tensors

`label` is stored as a HF `ClassLabel`, so `.names` gives the 200 class names in order. `concepts` is a single ground-truth vector column; its column order matches the 112 individual per-concept columns, which we use to recover `concept_names` -- so everything here comes straight from the loaded dataset, no local metadata files needed.

In [2]:
import torch

NON_CONCEPT_COLS = {
    "sample_idx", "split", "label", "class_name", "image_path", "image", "embedding", "concepts",
}

class_names = ds["train"].features["label"].names
concept_names = [c for c in ds["train"].column_names if c not in NON_CONCEPT_COLS]


def split_tensors(split):
    embeddings = torch.tensor(ds[split]["embedding"])
    # z-normalize each split independently (matches the original pipeline)
    embeddings = (embeddings - embeddings.mean(0, keepdim=True)) / embeddings.std(0, keepdim=True)
    concepts = torch.tensor(ds[split]["concepts"])
    labels = torch.tensor(ds[split]["label"])
    return embeddings, concepts, labels


train_embeddings, train_concepts, train_y = split_tensors("train")
val_embeddings, val_concepts, val_y = split_tensors("val")
test_embeddings, test_concepts, test_y = split_tensors("test")

print(len(class_names), "classes,", len(concept_names), "concepts")
print("train:", train_embeddings.shape, "val:", val_embeddings.shape, "test:", test_embeddings.shape)

200 classes, 112 concepts
train: torch.Size([4796, 768]) val: torch.Size([1198, 768]) test: torch.Size([5794, 768])


## Step 1 -- CLIP embedding -> concept classifier

One SVM per concept (`concept_mask`, chosen for being discriminative between the two sparrow species), trained on every class *except* the hard pair, then evaluated on the hard pair's test samples to see how well concepts learned elsewhere transfer to it.

In [3]:
import numpy as np
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report

concept_mask = [23, 44, 48, 69, 89, 103]  # striped breast, buff breast, white throat, buff nape, solid belly, brown crown
SPARROW_PAIR = [123, 126]  # Le Conte Sparrow, Savannah Sparrow
masked_concept_names = [concept_names[i] for i in concept_mask]
print(masked_concept_names)

concept_model = MultiOutputClassifier(SVC(kernel="rbf", C=1.0, class_weight="balanced"))

train_other_subset = np.where(~np.isin(train_y.numpy(), SPARROW_PAIR))[0]
test_pair_subset = np.where(np.isin(test_y.numpy(), SPARROW_PAIR))[0]

concept_model.fit(
    train_embeddings[train_other_subset].numpy(),
    train_concepts[train_other_subset][:, concept_mask].numpy(),
)

concept_preds = concept_model.predict(test_embeddings[test_pair_subset].numpy())
print(classification_report(
    test_concepts[test_pair_subset][:, concept_mask].numpy(),
    concept_preds,
    target_names=masked_concept_names,
))

['striped breast', 'buff breast', 'white throat', 'buff nape', 'solid belly', 'brown crown']


                precision    recall  f1-score   support

striped breast       0.54      0.97      0.69        30
   buff breast       0.67      0.97      0.79        29
  white throat       0.82      0.77      0.79        30
     buff nape       0.41      0.59      0.49        29
   solid belly       0.93      0.45      0.60        29
   brown crown       0.50      0.30      0.38        30

     micro avg       0.60      0.67      0.64       177
     macro avg       0.64      0.67      0.62       177
  weighted avg       0.64      0.67      0.62       177
   samples avg       0.63      0.67      0.63       177



## Step 2 -- concept -> label classifier (ground-truth concepts, hard pair only)

Logistic regression trained only on the two confusable species, using their *true* concept values (mapped to +/-1) -- an upper bound on how separable the pair is given perfect concept detection.

In [4]:
from sklearn.linear_model import LogisticRegression

train_pair_subset = np.where(np.isin(train_y.numpy(), SPARROW_PAIR))[0]
train_pair_concepts = train_concepts[train_pair_subset][:, concept_mask].numpy()
train_pair_y = train_y[train_pair_subset].numpy()

task_model = LogisticRegression(max_iter=1000, class_weight="balanced", fit_intercept=False)
task_model.fit(2 * train_pair_concepts - 1, train_pair_y)

test_pair_concepts = test_concepts[test_pair_subset][:, concept_mask].numpy()
test_pair_y = test_y[test_pair_subset].numpy()

label_preds = task_model.predict(2 * test_pair_concepts - 1)
print(classification_report(test_pair_y, label_preds))

              precision    recall  f1-score   support

         123       1.00      1.00      1.00        29
         126       1.00      1.00      1.00        30

    accuracy                           1.00        59
   macro avg       1.00      1.00      1.00        59
weighted avg       1.00      1.00      1.00        59



## Step 3 -- end-to-end: image -> predicted concepts -> predicted label

Chains the two fitted models: CLIP embeddings go through the concept SVMs (`tanh` of the decision function, a soft/continuous concept activation) and then through the logistic regression -- this is the pipeline a real user (or user-study participant) would actually see.

In [5]:
logits = [estimator.decision_function(test_embeddings[test_pair_subset].numpy()) for estimator in concept_model.estimators_]
concept_activations = np.tanh(np.column_stack(logits))

y_pred = task_model.predict(concept_activations)
print("=== END-TO-END (image -> predicted concepts -> predicted label) ===")
print(classification_report(test_pair_y, y_pred))

weights = task_model.coef_[0]
intercept = task_model.intercept_[0]
print("weights:", weights)
print("intercept:", intercept)

=== END-TO-END (image -> predicted concepts -> predicted label) ===
              precision    recall  f1-score   support

         123       0.78      0.86      0.82        29
         126       0.85      0.77      0.81        30

    accuracy                           0.81        59
   macro avg       0.82      0.81      0.81        59
weighted avg       0.82      0.81      0.81        59

weights: [ 0.70998989 -0.70998989  0.70998989 -0.70998989 -0.70998989  0.70998989]
intercept: 0.0


## Step 4 -- tidy per-sample CSV (ground truth vs. predicted, for the user study)

In [6]:
import pandas as pd
from pathlib import Path

test_pair_species = [class_names[label] for label in test_pair_y]
concept_activations_binary = (concept_activations > 0).astype(int)
concept_gt = test_concepts[test_pair_subset][:, concept_mask].numpy()


def build_df(gt, activations, activations_binary, labels, y_preds, species, concept_names, original_idx, split):
    base = pd.DataFrame({
        "split": split,
        "sample_idx": original_idx,
        "label": labels,
        "species": species,
        "task_pred": y_preds,
    })
    gt_df = pd.DataFrame(gt, columns=[f"{c}_gt" for c in concept_names])
    # _pred columns hold the continuous tanh activation (used again below as the
    # "legible" concept signal a human would see), _correct compares it binarized
    pred_df = pd.DataFrame(activations, columns=[f"{c}_pred" for c in concept_names])
    correct_df = pd.DataFrame((gt == activations_binary).astype(int), columns=[f"{c}_correct" for c in concept_names])
    return pd.concat([base, gt_df, pred_df, correct_df], axis=1)


test_df = build_df(
    concept_gt, concept_activations, concept_activations_binary, test_pair_y, y_pred,
    test_pair_species, masked_concept_names, test_pair_subset, split="test",
)
test_df.head()

,split,sample_idx,label,species,task_pred,striped breast_gt,buff breast_gt,white throat_gt,buff nape_gt,solid belly_gt,...,white throat_pred,buff nape_pred,solid belly_pred,brown crown_pred,striped breast_correct,buff breast_correct,white throat_correct,buff nape_correct,solid belly_correct,brown crown_correct
0,test,3520,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.866693,-0.387743,-0.599325,-0.534472,0,1,1,0,0,1
1,test,3521,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.843594,0.687462,-0.341700,-0.418174,0,1,1,1,0,1
2,test,3522,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.674573,0.769137,-0.528233,0.346006,0,1,1,1,0,0
3,test,3523,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.901123,0.624179,-0.165908,-0.736422,0,1,1,1,0,1
4,test,3524,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.762420,0.038375,0.241697,-0.313701,0,1,1,1,1,1


In [7]:
output_path = Path("../data/user_study/cub_user_study.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
test_df.to_csv(output_path, index=False)
print(f"Saved {len(test_df)} rows to {output_path}")

Saved 59 rows to ../data/user_study/cub_user_study.csv


## Step 5 -- toy interpretable formula check

A plain weighted sum over the (already fitted) concept activations, using `task_model`'s own coefficients/intercept -- this is the formula a human user-study participant would compute by hand. It should reproduce the model's predictions exactly.

In [8]:
dataframe = pd.read_csv(output_path)

pred_cols = [f"{c}_pred" for c in masked_concept_names]
p_concepts = dataframe[pred_cols]


def logistic(x):
    return 1 / (1 + np.exp(-x))


def compute_prediction(activations, betas, bias):
    eta = bias
    for i, b in enumerate(betas):
        eta += b * activations.iloc[:, i]
    return logistic(eta)


toy_model = compute_prediction(p_concepts, weights, intercept)
toy_predictions = np.where(toy_model > 0.5, task_model.classes_[1], task_model.classes_[0])

diff = toy_predictions != dataframe["task_pred"].to_numpy()
print(f"Differences between the hand-computed formula and the fitted model: {diff.sum()} / {len(diff)}")

Differences between the hand-computed formula and the fitted model: 0 / 59
